# PHASE 2: Data Cleaning & Preprocessing

In [7]:
import pandas as pd
import numpy as np

# Load datasets
train_df = pd.read_csv("../data/raw/kdd_train.csv")
test_df = pd.read_csv("../data/raw/kdd_test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (125973, 42)
Test shape: (22544, 42)


## Step 2.1 — Remove Duplicates from Training Set

In [ ]:
# Remove duplicates from training only (never from test)
train_df = train_df.drop_duplicates()

print("Train after deduplication:", train_df.shape)

Train after deduplication: (125973, 42)


## Step 2.2 — Fix the 23 vs 36 Class Problem (Critical!)

In [13]:
# Combine train + test, then re-split with stratification
# This ensures all classes can be represented in both sets

combined_df = pd.concat(
    [train_df, test_df],
    axis=0,
    ignore_index=True
)

print("Combined shape:", combined_df.shape)
print("Total unique classes:", combined_df["labels"].nunique())

# Check class counts in combined data
class_counts = combined_df["labels"].value_counts()

print("\nClass counts (combined):")
print(class_counts)

# Identify classes with very few samples
rare_classes = class_counts[class_counts < 5]

print(
    f"\nClasses with < 5 samples ({len(rare_classes)}):"
)
print(rare_classes)

Combined shape: (148517, 42)
Total unique classes: 37

Class counts (combined):
labels
normal             78588
neptune            47868
satan               4331
ipsweep             4078
portsweep           3302
smurf               3186
nmap                1699
back                1183
warezclient          997
teardrop             996
guess_passwd         464
mscan                310
warezmaster          299
pod                  236
apache2              228
processtable         211
snmpguess             99
mailbomb              94
saint                 93
buffer_overflow       47
snmpgetattack         43
httptunnel            41
land                  20
multihop              16
rootkit               14
imap                  13
loadmodule            13
ftp_write             10
ps                     9
sendmail               8
phf                    5
perl                   4
xlock                  4
xterm                  3
spy                    2
named                  2
xsnoop       

## Step 2.3 — Handle Extremely Rare Classes

In [21]:
# Strategy: Keep classes with at least 10 samples
# Classes with fewer than 10 samples are too rare for reliable
# stratified train/test splitting.

min_samples = 10

valid_classes = class_counts[
    class_counts >= min_samples
].index.tolist()

print(
    f"Classes with >= {min_samples} samples: "
    f"{len(valid_classes)}"
)

# Filter combined dataset
combined_filtered = combined_df[
    combined_df["labels"].isin(valid_classes)
].copy()

print("Filtered combined shape:", combined_filtered.shape)
print(
    "Remaining classes:",
    combined_filtered["labels"].nunique()
)

# Show removed rare classes
removed_classes = class_counts[
    class_counts < min_samples
]

print(
    f"\nRemoved classes with < {min_samples} samples "
    f"({len(removed_classes)}):"
)
print(removed_classes)

Classes with >= 10 samples: 28
Filtered combined shape: (148479, 42)
Remaining classes: 28

Removed classes with < 10 samples (9):
labels
ps          9
sendmail    8
phf         5
perl        4
xlock       4
xterm       3
spy         2
named       2
xsnoop      1
Name: count, dtype: int64


## Step 2.4 — Stratified Train/Test Split

In [28]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = combined_filtered.drop("labels", axis=1)
y = combined_filtered["labels"]

# Stratified split:
# Keeps approximately the same class proportion in train and test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

print("\nTrain class distribution:")
print(y_train.value_counts())

print("\nTest class distribution:")
print(y_test.value_counts())

# Verify all test classes also exist in training
train_classes = set(y_train.unique())
test_classes = set(y_test.unique())

missing_in_train = test_classes - train_classes

print(
    f"\nClasses in test but not in train: "
    f"{len(missing_in_train)}"
)

if missing_in_train:
    print(missing_in_train)

Training set shape: (118783, 41)
Testing set shape: (29696, 41)

Train class distribution:
labels
normal             62870
neptune            38294
satan               3465
ipsweep             3262
portsweep           2642
smurf               2549
nmap                1359
back                 946
warezclient          798
teardrop             797
guess_passwd         371
mscan                248
warezmaster          239
pod                  189
apache2              182
processtable         169
snmpguess             79
mailbomb              75
saint                 74
buffer_overflow       38
snmpgetattack         34
httptunnel            33
land                  16
multihop              13
imap                  11
rootkit               11
loadmodule            11
ftp_write              8
Name: count, dtype: int64

Test class distribution:
labels
normal             15718
neptune             9574
satan                866
ipsweep              816
portsweep            660
smurf             

## Step 2.5 — Encode Categorical Features

In [34]:
from sklearn.preprocessing import LabelEncoder

# Categorical feature columns
categorical_cols = [
    "protocol_type",
    "service",
    "flag"
]

# Target column
target_col = "labels"

# Initialize encoders
feature_encoders = {}
target_encoder = LabelEncoder()

# Create copies
X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

# Encode categorical features
for col in categorical_cols:
    le = LabelEncoder()

    # IMPORTANT:
    # Learn the mapping from TRAINING data only
    le.fit(X_train[col])

    # Apply the SAME mapping to train and test
    X_train_encoded[col] = le.transform(X_train[col])
    X_test_encoded[col] = le.transform(X_test[col])

    feature_encoders[col] = le

    print(f"{col}: {len(le.classes_)} categories encoded")

    # Show exact mapping
    print("Mapping:")
    for i, category in enumerate(le.classes_):
        print(f"  {i} → {category}")

# Encode target labels
target_encoder.fit(y_train)

y_train_encoded = target_encoder.transform(y_train)
y_test_encoded = target_encoder.transform(y_test)

print(
    f"\nTarget classes ({len(target_encoder.classes_)}):"
)

for i, cls in enumerate(target_encoder.classes_):
    print(f"  {i} → {cls}")

protocol_type: 3 categories encoded
Mapping:
  0 → icmp
  1 → tcp
  2 → udp
service: 70 categories encoded
Mapping:
  0 → IRC
  1 → X11
  2 → Z39_50
  3 → aol
  4 → auth
  5 → bgp
  6 → courier
  7 → csnet_ns
  8 → ctf
  9 → daytime
  10 → discard
  11 → domain
  12 → domain_u
  13 → echo
  14 → eco_i
  15 → ecr_i
  16 → efs
  17 → exec
  18 → finger
  19 → ftp
  20 → ftp_data
  21 → gopher
  22 → harvest
  23 → hostnames
  24 → http
  25 → http_2784
  26 → http_443
  27 → http_8001
  28 → imap4
  29 → iso_tsap
  30 → klogin
  31 → kshell
  32 → ldap
  33 → link
  34 → login
  35 → mtp
  36 → name
  37 → netbios_dgm
  38 → netbios_ns
  39 → netbios_ssn
  40 → netstat
  41 → nnsp
  42 → nntp
  43 → ntp_u
  44 → other
  45 → pm_dump
  46 → pop_2
  47 → pop_3
  48 → printer
  49 → private
  50 → red_i
  51 → remote_job
  52 → rje
  53 → shell
  54 → smtp
  55 → sql_net
  56 → ssh
  57 → sunrpc
  58 → supdup
  59 → systat
  60 → telnet
  61 → tftp_u
  62 → tim_i
  63 → time
  64 → urh_i
  

## Step 2.6 — Save Encoding Mappings

In [39]:
import json
import os

# Create processed-data directory if it doesn't exist
os.makedirs("../data/processed", exist_ok=True)

# Save feature mappings
mapping = {}

for col, le in feature_encoders.items():
    mapping[col] = {
        str(i): cls
        for i, cls in enumerate(le.classes_)
    }

# Save target mapping
mapping["labels"] = {
    str(i): cls
    for i, cls in enumerate(target_encoder.classes_)
}

# Save mapping file
mapping_path = "../data/processed/encoding_mapping.json"

with open(mapping_path, "w") as f:
    json.dump(mapping, f, indent=2)

print("Encoding mapping saved to:", mapping_path)

Encoding mapping saved to: ../data/processed/encoding_mapping.json


## Step 2.7 — Feature Scaling (Model-Dependent)

In [43]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# For distance-based models (KNN, SVM, Neural Networks)
# For tree-based models (Random Forest, XGBoost), scaling is optional

scaler = StandardScaler()

# Fit on training data ONLY, then transform both
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

print("Scaling complete.")
print("Train scaled shape:", X_train_scaled.shape)
print("Test scaled shape:", X_test_scaled.shape)

Scaling complete.
Train scaled shape: (118783, 41)
Test scaled shape: (29696, 41)


## Step 2.8 — Save Preprocessed Data

In [47]:
import joblib
import os

# Create processed-data directory if it doesn't exist
os.makedirs("../data/processed", exist_ok=True)

# Save encoded features
X_train_encoded.to_csv(
    "../data/processed/X_train_encoded.csv",
    index=False
)

X_test_encoded.to_csv(
    "../data/processed/X_test_encoded.csv",
    index=False
)

# Save encoded target
np.save(
    "../data/processed/y_train_encoded.npy",
    y_train_encoded
)

np.save(
    "../data/processed/y_test_encoded.npy",
    y_test_encoded
)

# Save scaler and encoders
joblib.dump(
    scaler,
    "../data/processed/scaler.pkl"
)

joblib.dump(
    feature_encoders,
    "../data/processed/feature_encoders.pkl"
)

joblib.dump(
    target_encoder,
    "../data/processed/target_encoder.pkl"
)

print("Phase 2 complete: Data cleaned, encoded, and saved.")

Phase 2 complete: Data cleaned, encoded, and saved.
